## Setup 
This is the setup I used to run this example, you might have to install, bind & connect things differently. :) </br></br>

1. **Install & start Qdrant vector database locally**
I'm running this code inside a docker container and not able to run qdrant via docker command.
Therefore, I ran the qdrant container on the same level as the container running this code & connect to the qdrant container using my server's IP.

```bash
docker pull qdrant/qdrant
docker run -p 6333:6333 qdrant/qdrant
```
Access Qdrant's interactive Web UI at http://localhost:6333/dashboard

2. **Install Ollama and pull llama3.1 model**
In the container running my code, I installed `ollama` directly
```bash
$ curl -fsSL https://ollama.com/install.sh | sh

# Download llama3.1 model
$ ollama pull llama3.1
pulling manifest 
pulling 667b0c1932bc... 100% ▕████████████████████▏ 4.9 GB                         
pulling 948af2743fc7... 100% ▕████████████████████▏ 1.5 KB                         
pulling 0ba8f0e314b4... 100% ▕████████████████████▏  12 KB                         
pulling 56bb8bd477a5... 100% ▕████████████████████▏   96 B                         
pulling 455f34728c9b... 100% ▕████████████████████▏  487 B                         
verifying sha256 digest 
writing manifest 
success 

# Check downloaded model
$ ollama list
NAME               ID              SIZE      MODIFIED    
llama3.1:latest    46e0c10c039e    4.9 GB    3 hours ago

# Run llama3.1 model
$ ollama run llama3.1
>>> Send a message (/? for help)
```

In [1]:
!pip install mem0ai

DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/opt_einsum-3.4.0-py3.12.egg is deprecated. pip 25.1 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/dill-0.3.9-py3.12.egg is deprecated. pip 25.1 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/nvfuser-0.2.13a0+0d33366-py3.12-linux-x86_64.egg is deprecated. pip 25.1 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/texttable-1.7.0-py3.12.egg is deprecated. pip 25.1 will 

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

config = {
    "vector_store": {
        "provider": "qdrant",
        "config": {
            "collection_name": "local-chatgpt-memory",
            "host": os.environ.get("QDRANT_HOST", "127.0.0.1"),
            "port": 6333,
            "embedding_model_dims": 768,
        },
    },
    "llm": {
        "provider": "ollama",
        "config": {
            "model": "llama3.1:latest",
            "temperature": 0,
            "max_tokens": 8000,
            "ollama_base_url": f"http://localhost:11434",  # Ensure this URL is correct
        },
    },
    "embedder": {
        "provider": "ollama",
        "config": {
            "model": "nomic-embed-text:latest",
            # Alternatively, you can use "snowflake-arctic-embed:latest"
            "ollama_base_url": "http://localhost:11434",
        },
    },
    "version": "v1.1"
}

In [2]:
from mem0 import Memory

memory = Memory.from_config(config)

I'm curious of what was going on behind the scene when adding ```messages``` to the memory. </br>
So I checked the package [code](https://github.com/mem0ai/mem0/blob/main/mem0/memory/main.py) and here's what I found out.
1. Having a prompt from the user below saying "I'm visiting Paris".
2. mem0's Memory() processes the prompt by 
    1. Parse the prompt to see if the prompt is a conversation between the user and the assistant (the conversation example is demonstrated later in the notebook).
        ```javascript
        198     parsed_messages = parse_messages(messages)
        ```
    2. mem0 has a few constant system's prompts they defined [here](https://github.com/mem0ai/mem0/blob/main/mem0/configs/prompts.py) to instruct the llm (set in the config) to retrieve facts from the parsed prompt.
        - As of 2025/04/16, there are 4 Constants: `MEMORY_ANSWER_PROMPT`, `FACT_RETRIEVAL_PROMPT`, `DEFAULT_UPDATE_MEMORY_PROMPT`, `PROCEDURAL_MEMORY_SYSTEM_PROMPT`
        
        ```javascript 
        204     system_prompt, user_prompt = get_fact_retrieval_messages(parsed_messages)
        ```
    3. The System, User & Assistant's Prompt are passed to the LLM for "fact retrieval" inference.
        ```javascript
        206     response = self.llm.generate_response(
        207         messages=[
        208             {"role": "system", "content": system_prompt},
        209             {"role": "user", "content": user_prompt},
        210        ],
        211         response_format={"type": "json_object"},
        212     )
        ```
    4. The LLM (here is llama3.1) generated a response in the form of **JSON**, which is then accessed on line `216`. </br> 
    However, this code access is not general for all LLMs. I encountered `error` when using LangChain's ChatHuggingFace model because the llm's output cannot be transformed to **JSON**.
        ```javascript
        214     try:
        215         response = remove_code_blocks(response)
        216         new_retrieved_facts = json.loads(response)["facts"]
        217     except Exception as e:
        218         logging.error(f"Error in new_retrieved_facts: {e}")
        219         new_retrieved_facts = []
        ```

In [ ]:
# Example 1
memory.add("I'm visiting Paris", user_id="john")

***** john - None - None *****
[{'role': 'user', 'content': "I'm visiting Paris"}]
1*****************************************
[{'role': 'user', 'content': "I'm visiting Paris"}]
2*****************************************
[{'role': 'user', 'content': "I'm visiting Paris"}]
{'user_id': 'john'}
{'user_id': 'john'}
True
3*****************************************
{"facts" : ["Visiting Paris"]}
4*****************************************
{"facts" : ["Visiting Paris"]}
4*****************************************
['Visiting Paris']
5*****************************************


{'results': [{'id': 'b0ca0cf0-a617-4520-9527-86f837edf445',
   'memory': 'Visiting Paris',
   'event': 'ADD'}]}

In [6]:
memories = memory.get_all(user_id="john")
memories

{'results': [{'id': 'b0ca0cf0-a617-4520-9527-86f837edf445',
   'memory': 'Visiting Paris',
   'hash': '7326ebb98166c98f0e32fdb5fc065a4b',
   'metadata': None,
   'created_at': '2025-04-15T22:26:53.041563-07:00',
   'updated_at': None,
   'user_id': 'john'}]}

In [ ]:
# Example 2
messages = [
    {"role": "user", "content": "I'm planning to watch a movie tonight. Any recommendations?"},
    {"role": "assistant", "content": "How about a thriller movies? They can be quite engaging."},
    {"role": "user", "content": "I'm not a big fan of thriller movies but I love sci-fi movies."},
    {"role": "assistant", "content": "Got it! I'll avoid thriller recommendations and suggest sci-fi movies in the future."}
]

memory.add(messages, user_id="alice", metadata={"category": "movies"})

***** alice - None - None *****
[{'role': 'user', 'content': "I'm planning to watch a movie tonight. Any recommendations?"}, {'role': 'assistant', 'content': 'How about a thriller movies? They can be quite engaging.'}, {'role': 'user', 'content': "I'm not a big fan of thriller movies but I love sci-fi movies."}, {'role': 'assistant', 'content': "Got it! I'll avoid thriller recommendations and suggest sci-fi movies in the future."}]
1*****************************************
[{'role': 'user', 'content': "I'm planning to watch a movie tonight. Any recommendations?"}, {'role': 'assistant', 'content': 'How about a thriller movies? They can be quite engaging.'}, {'role': 'user', 'content': "I'm not a big fan of thriller movies but I love sci-fi movies."}, {'role': 'assistant', 'content': "Got it! I'll avoid thriller recommendations and suggest sci-fi movies in the future."}]
2*****************************************
[{'role': 'user', 'content': "I'm planning to watch a movie tonight. Any r

{'results': [{'id': 'dd2c9324-a39f-478c-8ae2-734d29a1cecd',
   'memory': 'Not a big fan of thriller movies',
   'event': 'ADD'},
  {'id': '2bf3b1da-4dbd-4f40-8634-d3efc4b46a25',
   'memory': 'Love sci-fi movies',
   'event': 'ADD'}]}

In [7]:
memories = memory.get_all(user_id="alice")
memories

{'results': [{'id': '2bf3b1da-4dbd-4f40-8634-d3efc4b46a25',
   'memory': 'Love sci-fi movies',
   'hash': '65ef203273b0fa207b3134c45a097349',
   'metadata': {'category': 'movies'},
   'created_at': '2025-04-15T22:29:53.595830-07:00',
   'updated_at': None,
   'user_id': 'alice'},
  {'id': 'dd2c9324-a39f-478c-8ae2-734d29a1cecd',
   'memory': 'Not a big fan of thriller movies',
   'hash': '028dfab4483f28980e292f62578d3293',
   'metadata': {'category': 'movies'},
   'created_at': '2025-04-15T22:29:53.590979-07:00',
   'updated_at': None,
   'user_id': 'alice'}]}